In [1]:
# Завантаження датасету через urllib
import urllib.request
import zipfile
import os

def download_power_dataset():
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip'
    zip_path = 'power_data/household_power_consumption.zip'
    data_path = 'power_data/household_power_consumption.txt'
    
    if not os.path.exists('power_data'):
        os.makedirs('power_data')
    
    if os.path.exists(data_path):
        print(f"Файл вже існує: {data_path}")
        return data_path
    
    print("Завантаження датасету...")
    urllib.request.urlretrieve(url, zip_path)
    print("✓ Завантажено, розпаковуємо...")
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('power_data')
    
    os.remove(zip_path)
    print(f"✓ Готово: {data_path}")
    return data_path

data_path = download_power_dataset()

Завантаження датасету...
✓ Завантажено, розпаковуємо...
✓ Готово: power_data/household_power_consumption.txt


In [2]:
# Імпорт бібліотек
import pandas as pd
import numpy as np
import timeit
import os
import urllib.request
import zipfile

print("Бібліотеки імпортовані!")

Бібліотеки імпортовані!


In [3]:
# Читання датасету
df = pd.read_csv(data_path, sep=';', low_memory=False)
print(f"Розмір: {df.shape}")
print(f"\nПерші рядки:")
print(df.head())
print(f"\nТипи даних:\n{df.dtypes}")
print(f"\nПропуски:\n{df.isnull().sum()}")

Розмір: (2075259, 9)

Перші рядки:
         Date      Time Global_active_power Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00               4.216                 0.418  234.840   
1  16/12/2006  17:25:00               5.360                 0.436  233.630   
2  16/12/2006  17:26:00               5.374                 0.498  233.290   
3  16/12/2006  17:27:00               5.388                 0.502  233.740   
4  16/12/2006  17:28:00               3.666                 0.528  235.680   

  Global_intensity Sub_metering_1 Sub_metering_2  Sub_metering_3  
0           18.400          0.000          1.000            17.0  
1           23.000          0.000          1.000            16.0  
2           23.000          0.000          2.000            17.0  
3           23.000          0.000          1.000            17.0  
4           15.800          0.000          1.000            17.0  

Типи даних:
Date                         str
Time                         str
Global_active_p

In [4]:
# Data cleaning
def clean_power_data(df):
    # Замінюємо '?' на NaN
    df = df.replace('?', pd.NA)
    
    # Об'єднуємо Date і Time в один datetime стовпець
    df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
    df = df.drop(['Date', 'Time'], axis=1)
    
    # Переміщуємо Datetime на початок
    cols = ['Datetime'] + [c for c in df.columns if c != 'Datetime']
    df = df[cols]
    
    # Конвертуємо всі числові стовпці
    num_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage',
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Заповнюємо пропуски середнім значенням по кожному стовпцю
    df[num_cols] = df[num_cols].fillna(df[num_cols].mean())
    
    return df

df = clean_power_data(df)

print(f"Розмір після cleaning: {df.shape}")
print(f"\nТипи даних:\n{df.dtypes}")
print(f"\nПропуски:\n{df.isnull().sum()}")
print(f"\nПерші рядки:")
print(df.head())

Розмір після cleaning: (2075259, 8)

Типи даних:
Datetime                 datetime64[us]
Global_active_power             float64
Global_reactive_power           float64
Voltage                         float64
Global_intensity                float64
Sub_metering_1                  float64
Sub_metering_2                  float64
Sub_metering_3                  float64
dtype: object

Пропуски:
Datetime                 0
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64

Перші рядки:
             Datetime  Global_active_power  Global_reactive_power  Voltage  \
0 2006-12-16 17:24:00                4.216                  0.418   234.84   
1 2006-12-16 17:25:00                5.360                  0.436   233.63   
2 2006-12-16 17:26:00                5.374                  0.498   233.29   
3 2006-12-16 17:27:00                5.388         

In [5]:
# Вибірка 1: записи де Global_active_power > 5 кВт
def select_high_power(df):
    result = df[df['Global_active_power'] > 5]
    print(f"Записи з активною потужністю > 5 кВт: {len(result)}")
    print(result.head())
    return result

result1 = select_high_power(df);

Записи з активною потужністю > 5 кВт: 17547
              Datetime  Global_active_power  Global_reactive_power  Voltage  \
1  2006-12-16 17:25:00                5.360                  0.436   233.63   
2  2006-12-16 17:26:00                5.374                  0.498   233.29   
3  2006-12-16 17:27:00                5.388                  0.502   233.74   
11 2006-12-16 17:35:00                5.412                  0.470   232.78   
12 2006-12-16 17:36:00                5.224                  0.478   232.99   

    Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
1               23.0             0.0             1.0            16.0  
2               23.0             0.0             2.0            17.0  
3               23.0             0.0             1.0            17.0  
11              23.2             0.0             1.0            17.0  
12              22.4             0.0             1.0            16.0  


In [6]:
# Вибірка 2: сила струму 19-20А, пральна+холодильник > бойлер+кондиціонер
def select_by_current_and_appliances(df):
    # Global_intensity між 19 і 20
    filtered = df[(df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20)]
    
    # Sub_metering_2 (пральна+холодильник) > Sub_metering_3 (бойлер+кондиціонер)
    result = filtered[filtered['Sub_metering_2'] > filtered['Sub_metering_3']]
    
    print(f"Записи з струмом 19-20А: {len(filtered)}")
    print(f"З них пральна+холодильник > бойлер+кондиціонер: {len(result)}")
    print(result.head())
    return result

result2 = select_by_current_and_appliances(df);

Записи з струмом 19-20А: 7021
З них пральна+холодильник > бойлер+кондиціонер: 2509
               Datetime  Global_active_power  Global_reactive_power  Voltage  \
45  2006-12-16 18:09:00                4.464                  0.136   234.66   
460 2006-12-17 01:04:00                4.582                  0.258   238.08   
464 2006-12-17 01:08:00                4.618                  0.104   239.61   
475 2006-12-17 01:19:00                4.636                  0.140   237.37   
476 2006-12-17 01:20:00                4.634                  0.152   237.17   

     Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
45               19.0             0.0            37.0            16.0  
460              19.6             0.0            13.0             0.0  
464              19.6             0.0            27.0             0.0  
475              19.4             0.0            36.0             0.0  
476              19.4             0.0            35.0             0.0  


In [7]:
# Вибірка 3: 500000 випадкових записів, середні по групах споживання
def select_random_and_avg(df):
    sample = df.sample(n=500000, replace=False, random_state=42)
    
    avg = sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean().round(4)
    
    print(f"Розмір вибірки: {len(sample)}")
    print(f"\nСередні значення груп споживання:")
    print(f"  Sub_metering_1 (кухня):             {avg['Sub_metering_1']} Вт·год")
    print(f"  Sub_metering_2 (пральна+холодильник): {avg['Sub_metering_2']} Вт·год")
    print(f"  Sub_metering_3 (бойлер+кондиціонер): {avg['Sub_metering_3']} Вт·год")
    return sample

result3 = select_random_and_avg(df);

Розмір вибірки: 500000

Середні значення груп споживання:
  Sub_metering_1 (кухня):             1.123 Вт·год
  Sub_metering_2 (пральна+холодильник): 1.3 Вт·год
  Sub_metering_3 (бойлер+кондиціонер): 6.4654 Вт·год


In [8]:
# Вибірка 4: після 18:00, > 6 кВт, група 2 найбільша, кожен 3-й з першої половини і кожен 4-й з другої
def select_evening_high_power(df):
    # Після 18:00
    filtered = df[df['Datetime'].dt.hour >= 18]
    
    # Більше 6 кВт
    filtered = filtered[filtered['Global_active_power'] > 6]
    
    # Група 2 є найбільшою
    filtered = filtered[
        (filtered['Sub_metering_2'] > filtered['Sub_metering_1']) &
        (filtered['Sub_metering_2'] > filtered['Sub_metering_3'])
    ]
    
    # Ділимо на дві половини
    mid = len(filtered) // 2
    first_half = filtered.iloc[:mid].iloc[::3]   # кожен 3-й
    second_half = filtered.iloc[mid:].iloc[::4]  # кожен 4-й
    
    result = pd.concat([first_half, second_half])
    
    print(f"Після 18:00 і > 6 кВт: {len(filtered)}")
    print(f"З них група 2 найбільша: {len(filtered)}")
    print(f"Перша половина (кожен 3-й): {len(first_half)}")
    print(f"Друга половина (кожен 4-й): {len(second_half)}")
    print(f"Фінальна вибірка: {len(result)}")
    print(result.head())
    return result

result4 = select_evening_high_power(df);

Після 18:00 і > 6 кВт: 1061
З них група 2 найбільша: 1061
Перша половина (кожен 3-й): 177
Друга половина (кожен 4-й): 133
Фінальна вибірка: 310
                 Datetime  Global_active_power  Global_reactive_power  \
41    2006-12-16 18:05:00                6.052                  0.192   
44    2006-12-16 18:08:00                6.308                  0.116   
17494 2006-12-28 20:58:00                6.386                  0.374   
17498 2006-12-28 21:02:00                8.088                  0.262   
17501 2006-12-28 21:05:00                7.230                  0.152   

       Voltage  Global_intensity  Sub_metering_1  Sub_metering_2  \
41      232.93              26.2             0.0            37.0   
44      232.25              27.0             0.0            36.0   
17494   236.63              27.0             1.0            36.0   
17498   235.50              34.4             1.0            72.0   
17501   235.22              30.6             1.0            73.0   

       S

In [9]:
# Нормалізація та стандартизація
def normalize_and_standardize(df):
    num_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage',
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    
    # Нормалізація (Min-Max): значення від 0 до 1
    df_normalized = df.copy()
    for col in num_cols:
        min_val = df[col].min()
        max_val = df[col].max()
        df_normalized[col] = (df[col] - min_val) / (max_val - min_val)
    
    # Стандартизація (Z-score): середнє=0, std=1
    df_standardized = df.copy()
    for col in num_cols:
        mean_val = df[col].mean()
        std_val = df[col].std()
        df_standardized[col] = (df[col] - mean_val) / std_val
    
    print("Нормалізований датасет (перші 3 рядки):")
    print(df_normalized[num_cols].head(3).round(4))
    print(f"\nСтандартизований датасет (перші 3 рядки):")
    print(df_standardized[num_cols].head(3).round(4))
    
    return df_normalized, df_standardized

df_norm, df_std = normalize_and_standardize(df);

Нормалізований датасет (перші 3 рядки):
   Global_active_power  Global_reactive_power  Voltage  Global_intensity  \
0               0.3748                 0.3007   0.3761            0.3776   
1               0.4784                 0.3137   0.3370            0.4730   
2               0.4796                 0.3583   0.3260            0.4730   

   Sub_metering_1  Sub_metering_2  Sub_metering_3  
0             0.0          0.0125          0.5484  
1             0.0          0.0125          0.5161  
2             0.0          0.0250          0.5484  

Стандартизований датасет (перші 3 рядки):
   Global_active_power  Global_reactive_power  Voltage  Global_intensity  \
0               2.9737                 2.6272  -1.8635            3.1184   
1               4.0626                 2.7879  -2.2393            4.1599   
2               4.0759                 3.3414  -2.3449            4.1599   

   Sub_metering_1  Sub_metering_2  Sub_metering_3  
0         -0.1835         -0.0516          1.25

In [11]:
import subprocess
subprocess.run(['pip', 'install', 'scipy', '--break-system-packages'])

CompletedProcess(args=['pip', 'install', 'scipy', '--break-system-packages'], returncode=0)

In [12]:
# Коефіцієнти Пірсона та Спірмена
from scipy import stats

def compute_correlations(df):
    col1 = 'Global_active_power'
    col2 = 'Global_intensity'
    
    pearson_r, pearson_p = stats.pearsonr(df[col1], df[col2])
    spearman_r, spearman_p = stats.spearmanr(df[col1], df[col2])
    
    print(f"Кореляція між '{col1}' та '{col2}':")
    print(f"\n  Пірсон:  r = {pearson_r:.4f},  p-value = {pearson_p:.4e}")
    print(f"  Спірмен: r = {spearman_r:.4f},  p-value = {spearman_p:.4e}")

compute_correlations(df);

Кореляція між 'Global_active_power' та 'Global_intensity':

  Пірсон:  r = 0.9989,  p-value = 0.0000e+00
  Спірмен: r = 0.9955,  p-value = 0.0000e+00


In [15]:
def one_hot_encoding(df):
    df = df.copy()
    df['time_of_day'] = df['Datetime'].dt.hour.apply(
        lambda h: 'ранок' if 6 <= h < 12 else 'день' if 12 <= h < 18 else 'вечір' if 18 <= h < 24 else 'ніч'
    )
    
    print("Розподіл категорій:")
    print(df['time_of_day'].value_counts())
    
    df_encoded = pd.get_dummies(df, columns=['time_of_day'], prefix='tod')
    ohe_cols = [c for c in df_encoded.columns if c.startswith('tod_')]
    df_encoded[ohe_cols] = df_encoded[ohe_cols].astype(int)
    
    print(f"\nНові стовпці після OHE:")
    print(df_encoded[ohe_cols].head(5))
    
    return df_encoded

df_encoded = one_hot_encoding(df);

Розподіл категорій:
time_of_day
вечір    518943
день     518796
ніч      518760
ранок    518760
Name: count, dtype: int64

Нові стовпці після OHE:
   tod_вечір  tod_день  tod_ніч  tod_ранок
0          0         1        0          0
1          0         1        0          0
2          0         1        0          0
3          0         1        0          0
4          0         1        0          0


In [14]:
# Профілювання часу виконання за допомогою timeit
import timeit

print("Час виконання функцій:\n")

t1 = timeit.timeit(lambda: select_high_power(df), number=1)
print(f"  select_high_power:              {t1:.4f} сек")

t2 = timeit.timeit(lambda: select_by_current_and_appliances(df), number=1)
print(f"  select_by_current_and_appliances: {t2:.4f} сек")

t3 = timeit.timeit(lambda: select_random_and_avg(df), number=1)
print(f"  select_random_and_avg:          {t3:.4f} сек")

t4 = timeit.timeit(lambda: select_evening_high_power(df), number=1)
print(f"  select_evening_high_power:      {t4:.4f} сек")

t5 = timeit.timeit(lambda: normalize_and_standardize(df), number=1)
print(f"  normalize_and_standardize:      {t5:.4f} сек")

Час виконання функцій:

Записи з активною потужністю > 5 кВт: 17547
              Datetime  Global_active_power  Global_reactive_power  Voltage  \
1  2006-12-16 17:25:00                5.360                  0.436   233.63   
2  2006-12-16 17:26:00                5.374                  0.498   233.29   
3  2006-12-16 17:27:00                5.388                  0.502   233.74   
11 2006-12-16 17:35:00                5.412                  0.470   232.78   
12 2006-12-16 17:36:00                5.224                  0.478   232.99   

    Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
1               23.0             0.0             1.0            16.0  
2               23.0             0.0             2.0            17.0  
3               23.0             0.0             1.0            17.0  
11              23.2             0.0             1.0            17.0  
12              22.4             0.0             1.0            16.0  
  select_high_power:           